# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mvdu12/ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding A -- 'What Predicts Health?' (Random Forest feature importance, ML Appendix, p.27)

**Label:** health_score. Per the paper's own metric guide (p.5), health_score is a CONSTRUCTED composite: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts).

**The reported top features:** Average Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8% -- these four are literally the four ingredients the label is built from. The paper does flag this ('the target itself is partly constructed from some of these inputs'), but still headlines 'Average Position is the #1 predictor of health score,' which reads like a discovery.

**My question:** does the holdout split rescue this claim? No -- holdout testing protects against overfitting to noise, but it can't rescue a model whose 'predictors' mathematically define the label. This is leakage taxonomy #1 (label-derived features), and no split design fixes it. A cleaner version of this finding would predict something health_score does NOT contain (e.g. future impression growth), or explicitly frame this as 'decomposing our own scoring formula' rather than 'predicting health.'

**Finding B** -- 'What Predicts Growth?' (Logistic Regression, 71% holdout accuracy, ML Appendix, p.29)
    
Label: growing vs. declining, from 30d-vs-prev-30d impression change. Features: content_age_days, days_since_last_update, days_visible, avg_position, word_count, impressions, search_volume, clicks, ai_sessions, sessions -- none of these are literally the label's ingredients, which is better than Finding A
    
My questions: (1) The paper never states whether the holdout split is grouped by brand (there are 57 brands in the portfolio) or purely random at the row level. Section 2 below shows this choice alone can swing my own equivalent metric by a huge margin. (2) '71% holdout accuracy' is reported with no base rate next to it -- Finding #1 (p.6) shows roughly 74.8K growing vs 45.6K declining pages in one cut of this portfolio, i.e. a majority class around 62%. If that holds for this appendix's 61.8K-row sample too, 71% is closer to a 9-point lift over guessing the majority class, not the standalone number it's presented as.

Neither of these is a fatal problem -- but naming the split design and the base rate would let a reader judge how much to trust the 71% figure, in the same spirit the paper already uses for its own myth-busting sections.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

url = 'https://raw.githubusercontent.com/Mvdu12/ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

numeric_feats = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
cat_feats = ['content_type', 'main_intent', 'competition_level']
X = df[numeric_feats + cat_feats].copy()
for c in numeric_feats:
    X[c + '_missing'] = X[c].isna().astype(int)
    X[c] = X[c].fillna(0)
X = pd.get_dummies(X, columns=cat_feats, dummy_na=True)
y = df['is_declining_label']
groups = df['client_id']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def run(train_idx, test_idx, label):
    Xtr, Xte = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
    rf.fit(Xtr, ytr)
    p = rf.predict_proba(Xte)[:, 1]
    res = {k: round(precision_at_k(p, yte.values, k), 3) for k in [50, 200, 500]}
    print(f"{label:38s} precision@K={res}  base_rate={yte.mean():.3f}  n_test={len(test_idx):,}")
    return res

# BEFORE: random row-level split -- the "naive" way to split
tr_r, te_r = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42, stratify=y)
before = run(tr_r, te_r, "BEFORE -- random row-level split:")

# AFTER: grouped by client_id -- the honest way, matching Week-5's design
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_g, te_g = next(gss.split(X, y, groups))
after = run(tr_g, te_g, "AFTER -- grouped by client_id:")

print(
    f"\nBefore/after gap at K=500: {before[500]:.3f} -> {after[500]:.3f} "
    f"({before[500]-after[500]:+.3f}). The random split looked far stronger "
    f"purely because pages from the same client leaked into both train and test -- "
    f"the model was partly recognizing clients, not decline risk. The grouped "
    f"number is the one Week-5's comparison table actually used, and it's the "
    f"one I trust."
)


BEFORE -- random row-level split:      precision@K={50: np.float64(0.86), 200: np.float64(0.86), 500: np.float64(0.83)}  base_rate=0.542  n_test=6,000
AFTER -- grouped by client_id:         precision@K={50: np.float64(0.48), 200: np.float64(0.525), 500: np.float64(0.584)}  base_rate=0.511  n_test=6,163

Before/after gap at K=500: 0.830 -> 0.584 (+0.246). The random split looked far stronger purely because pages from the same client leaked into both train and test -- the model was partly recognizing clients, not decline risk. The grouped number is the one Week-5's comparison table actually used, and it's the one I trust.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

print("Same hunt as Week 3 (ML-05), re-run on this notebook's final feature set.\n")

def fit_auc(Xd, name):
    Xtr, Xte = Xd.iloc[tr_g].copy(), Xd.iloc[te_g].copy()
    ytr, yte = y.iloc[tr_g], y.iloc[te_g]
    num_cols = [c for c in Xd.columns if c in numeric_feats]
    sc = StandardScaler()
    Xtr[num_cols] = sc.fit_transform(Xtr[num_cols])
    Xte[num_cols] = sc.transform(Xte[num_cols])
    m = LogisticRegression(max_iter=1000, random_state=42)
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(yte, p)
    print(f"{name}: AUC = {auc:.4f}")
    return auc

auc_final = fit_auc(X, "Final feature set (no trend_direction/trend_pct/IDs)")

X_check = X.copy()
X_check['trend_pct'] = df['trend_pct'].fillna(0)
auc_leak = fit_auc(X_check, "Re-adding trend_pct as a sanity check")

print(
    f"\nFinal set stays honest: {auc_final:.3f} AUC, nowhere near the {auc_leak:.3f} "
    f"seen the instant the label-derived column sneaks back in -- same collapse "
    f"signature as Week 3, confirming this feature set is still clean.\n"
    f"No product flags (health_score, priority_score, action_type, refresh flags) "
    f"exist in this dataset to accidentally include, and the split stays grouped "
    f"by client_id throughout Section 2. Checklist: timeline drawn (Week-5/ML-05), "
    f"no label-derived columns (above), no product flags (n/a here), grouped "
    f"split (Section 2), base rate printed next to every metric (Section 2), "
    f"metrics computed out-of-fold on held-out test rows only (Section 2)."
)


Same hunt as Week 3 (ML-05), re-run on this notebook's final feature set.

Final feature set (no trend_direction/trend_pct/IDs): AUC = 0.5395
Re-adding trend_pct as a sanity check: AUC = 1.0000

Final set stays honest: 0.539 AUC, nowhere near the 1.000 seen the instant the label-derived column sneaks back in -- same collapse signature as Week 3, confirming this feature set is still clean.
No product flags (health_score, priority_score, action_type, refresh flags) exist in this dataset to accidentally include, and the split stays grouped by client_id throughout Section 2. Checklist: timeline drawn (Week-5/ML-05), no label-derived columns (above), no product flags (n/a here), grouped split (Section 2), base rate printed next to every metric (Section 2), metrics computed out-of-fold on held-out test rows only (Section 2).


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence, from the Week-5 model-vs-baseline writeup (ML-08):**

> Random Forest pulls further ahead as K grows, suggesting it's finding non-linear structure the linear model can't.

**Rewritten in safe language:**

> On this grouped test split, Random Forest's precision@K was directionally higher than Logistic Regression's at every K tested (e.g. 0.584 vs 0.542 at K=500) — an observed pattern consistent with, but not proof of, the model capturing non-linear relationships the linear model misses. This is a decision-support signal for picking a model, not a settled explanation.

**What changed:** "suggesting X" (a causal-sounding inference) became "an observed pattern consistent with, but not proof of, X" — and "pulls further ahead" (a general claim) became a specific measured number at a specific K, on a specific split, so the claim can't quietly generalize beyond what was actually tested.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.